## Notes

In [ ]:
# can travel for 1/4 sq km in 1hr - 1 flight
# each sq km is 18,000rs

In [ ]:
INDIA_PROJECTED_CRS = "24378"

## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import openpyxl

import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

In [ ]:
from gridsample.utils import save_shapefiles
# from gridsample.mapping.plot import create_interactive_map

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
OUTPUT_DATA_DIR = DATA_DIR / "01_processed" / "Goverment Buildings"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

CONSOLIDATED_DATA_PATH = (
    RAW_DATA_DIR / "government_buildings" / "Consolidated_data_09042025.xlsx"
)

In [ ]:
workbook = openpyxl.load_workbook(CONSOLIDATED_DATA_PATH)

## Utilities

In [ ]:
def is_cell_highlighted(cell):
    if cell.fill.fill_type == "solid":
        # Check if the cell has a background color (not white/none)
        if (
            cell.fill.start_color.rgb != "00000000"
            and cell.fill.start_color.rgb != "FFFFFFFF"
        ):
            return True
    return False


def get_highlighted_rows(worksheet):
    highlighted_rows = []

    for row_idx, row in enumerate(worksheet.iter_rows(min_row=2)):  # Skip header row
        if sum(is_cell_highlighted(cell) for cell in row) >= 24:
            highlighted_rows.append(row_idx)

    return highlighted_rows

## Seoni

In [ ]:
SAVE_DIR_SEONI = OUTPUT_DATA_DIR / "Seoni"
SAVE_DIR_SEONI.mkdir(parents=True, exist_ok=True)


worksheet_seoni = workbook["Seoni"]

df_seoni = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Seoni", header=0)
df_seoni

In [ ]:
rows_to_drop_seoni = get_highlighted_rows(worksheet_seoni)

filtered_rows_df_seoni = df_seoni.drop(index=rows_to_drop_seoni).reset_index(drop=True)

# Filter to only building names and lat / long
filtered_df_seoni = filtered_rows_df_seoni[
    [
        "Name of institutions/consumer as per billing/connection details",
        "Latitude of location of consumer's permises",
        "longitude of location of consumer's permises",
    ]
]
filtered_df_seoni.rename(
    columns={
        "Name of institutions/consumer as per billing/connection details": "Name",
        "Latitude of location of consumer's permises": "lat",
        "longitude of location of consumer's permises": "lon",
    },
    inplace=True,
)

filtered_df_seoni.dropna().reset_index(drop=True, inplace=True)

filtered_df_seoni.to_csv(SAVE_DIR_SEONI / "cleaned_data.csv", index=False)

## Dhar

In [ ]:
SAVE_DIR_DHAR = OUTPUT_DATA_DIR / "Dhar/"
SAVE_DIR_DHAR.mkdir(parents=True, exist_ok=True)

worksheet_dhar = workbook["Dhar"]

df_dhar = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Dhar", header=0)
df_dhar

In [ ]:
rows_to_drop = get_highlighted_rows(worksheet_dhar)

filtered_rows_df_dhar = df_dhar.drop(
    index=rows_to_drop + [0]  # drop the first row which seems to be meaningless
).reset_index(drop=True)

In [ ]:
filtered_rows_df_dhar[filtered_rows_df_dhar["Latitude & Longitude of premises"].isna()]

In [ ]:
filtered_df_dhar = filtered_rows_df_dhar[
    ["Consumer Name  (Name of Office)", "Latitude & Longitude of premises"]
]
filtered_df_dhar

In [ ]:
# First split by commas or ". " to get lats
filtered_df_dhar["lat"] = filtered_df_dhar["Latitude & Longitude of premises"].apply(
    lambda x: str(x).split(",")[0].split(". ")[0]
)

# Then split by "." because some numbers have double decimal points -_-
filtered_df_dhar["lat"] = (
    filtered_df_dhar["lat"]
    .apply(
        lambda x: x
        if len(x.split(".")) <= 2
        else x.split(".")[0] + "." + x.split(".")[1]
    )
    .astype(float)
)

# INSPECT MANUALLY BECAUSE THERE MAY STILL BE SOME INSANITY
filtered_df_dhar["lat"].tolist()


In [ ]:
# Rinse and repeat for lons
filtered_df_dhar["lon"] = filtered_df_dhar["Latitude & Longitude of premises"].apply(
    lambda x: str(x).split(", ")[-1]
)

# Look for specific cases of outliers and fix:
pattern1 = r"(\d{2}\.\d+,\d{2}\.\d+|\d{2}\.\d+\.\d{2}\.\d+)"
pattern1_matches = filtered_df_dhar.loc[
    filtered_df_dhar["lon"].str.contains(pattern1, regex=True, na=False), "lon"
]
filtered_df_dhar.loc[pattern1_matches.index, "lon"] = pattern1_matches.apply(
    lambda x: str(x).split(",")[-1]
)

pattern2 = r"(\d{2}\.\d+. \d{2}\.\d+|\d{2}\.\d+\.\d{2}\.\d+)"
pattern2_matches = filtered_df_dhar.loc[
    filtered_df_dhar["lon"].str.contains(pattern2, regex=True, na=False), "lon"
]
filtered_df_dhar.loc[pattern2_matches.index, "lon"] = pattern2_matches.apply(
    lambda x: str(x).split(". ")[-1]
)


pattern3 = r"(\d{2}\.\d+\.\d+)"
pattern3_matches = filtered_df_dhar.loc[
    filtered_df_dhar["lon"].str.contains(pattern3, regex=True, na=False), "lon"
]
filtered_df_dhar.loc[pattern3_matches.index, "lon"] = pattern3_matches.apply(
    lambda x: str(x).split(".")[0] + "." + str(x).split(".")[1]
)

pattern4 = r"^[^.\,]*$"
not_nans = filtered_df_dhar["lon"] != "nan"
pattern4_matches = filtered_df_dhar.loc[
    (not_nans) & (filtered_df_dhar["lon"].str.contains(pattern4, regex=True, na=False)),
    "lon",
]
filtered_df_dhar.loc[pattern4_matches.index, "lon"] = pattern4_matches.apply(
    lambda x: str(x)[:2] + "." + str(x)[2:]
)

pattern5 = r"(\d{2}\,\d+)"
pattern5_matches = filtered_df_dhar.loc[
    filtered_df_dhar["lon"].str.contains(pattern5, regex=True, na=False), "lon"
]
filtered_df_dhar.loc[pattern5_matches.index, "lon"] = pattern5_matches.apply(
    lambda x: str(x).replace(",", ".")
)


filtered_df_dhar.loc[filtered_df_dhar["lon"] == "  75. ....3224", "lon"] = "75.3224"

# convert to float
filtered_df_dhar["lon"] = filtered_df_dhar["lon"].astype(float)

# INSPECT MANUALLY BECAUSE THERE MAY STILL BE SOME INSANITY
filtered_df_dhar["lon"].tolist()

In [ ]:
filtered_df_dhar.drop(columns=["Latitude & Longitude of premises"]).dropna(
    axis=0
).rename(columns={"Consumer Name  (Name of Office)": "Name"}).to_csv(
    SAVE_DIR_DHAR / "cleaned_data.csv", index=False
)

In [ ]:
df_dhar[~df_dhar["Consumer Name  (Name of Office)"].isin(filtered_df_dhar["Consumer Name  (Name of Office)"].unique())]

## Chhindwara

In [ ]:
SAVE_DIR_CHHINDWARA = OUTPUT_DATA_DIR / "Chhindwara/"
SAVE_DIR_CHHINDWARA.mkdir(parents=True, exist_ok=True)

worksheet_chhindwara = workbook["Chhindwara"]

df_chhindwara = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Chhindwara", header=0)
df_chhindwara

In [ ]:
rows_to_drop = get_highlighted_rows(worksheet_chhindwara)

filtered_rows_df_chhindwara = df_chhindwara.drop(index=rows_to_drop).reset_index(
    drop=True
)
filtered_df_chhindwara = filtered_rows_df_chhindwara[
    [
        "Name of institutions/consumer as per billing/connection details",
        "Latitude/Langitude of location of consumer's permises",
        "IVRS/Consumer Code as per billing details",
    ]
]
filtered_df_chhindwara

In [ ]:
filtered_df_chhindwara[
    "Latitude/Langitude of location of consumer's permises"
].to_list()
filtered_df_chhindwara["lat"] = filtered_df_chhindwara[
    "Latitude/Langitude of location of consumer's permises"
].apply(lambda x: str(x).split(" ")[0])
filtered_df_chhindwara["lat"].to_list()

In [ ]:
filtered_df_chhindwara[~filtered_df_chhindwara["lat"].astype(float).between(2, 23)]